# T5: who answers for the heat, and a notice they can be sent

Runs on the team's IBM z17 LinuxONE guest, in the `heatahead-jupyter` kernel, from the
`~/heatahead/site` clone (task: `docs/tasks/T5-messaging-landlord.md`).

1. Pull the managing-agent and corporate-owner rows of HPD registration contacts
   (NYC Open Data `feu5-w2e2`) and join them to the owner already in `bbl_landlord.csv`
   (`web/build_contacts.py`).
2. Check the privacy rules: business addresses only, street addresses only for companies,
   no unit or apartment designator anywhere.
3. Rebuild `public/data/` for the Vercel site with the new fields (`web/build_assets.py`).
4. Draft the tenant notice and the landlord self-audit for every ranked building through
   the same code the site serves, and time it on s390x.

`hpd_contacts.csv` and `bbl_landlord.csv` feed the frozen model's portfolio features, so
they are read here and never rewritten. Nothing in this notebook sends a message.

In [ ]:
import os, platform, subprocess, sys, time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "web"))
import pandas as pd

print("machine :", platform.machine(), "|", platform.node())
print("python  :", platform.python_version(), "| pandas", pd.__version__)
print("repo    :", ROOT, "| git", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

## 1. Pull HPD registration contacts and join them to the owner of record

In [ ]:
import build_contacts as bc

t0 = time.time()
raw = bc.pull()                      # Agent + CorporateOwner rows, paged 50,000 at a time
contacts = bc.build(raw)
print(f"pull + join: {time.time() - t0:.1f} s on {platform.machine()}")
print("columns pulled:", list(raw.columns))
contacts[contacts.bbl == 2043190010].T   # Building 60119, the running example

## 2. Coverage on the 63,727 ranked buildings, and the privacy checks

A street address is shown only for a company: the managing agent's business address
when the agent files as a company, the owner's when the owner is a corporation. An agent
filed as a person shows a name and a city; the building's full registration stays one
click away on HPD Online.

In [ ]:
risk = pd.read_csv("model/out/season_risk_2026.csv", usecols=["bbl", "rank", "p_season"])
c = contacts.fillna({"agent": "", "agent_addr": "", "owner_addr": ""})
m = risk.merge(c, on="bbl", how="left")
m["boro"] = (m.bbl // 1_000_000_000).map({1: "Manhattan", 2: "Bronx", 3: "Brooklyn",
                                         4: "Queens", 5: "Staten Island"})
reg = m.registrationid.notna()
cov = pd.DataFrame({
    "ranked buildings": m.groupby("boro").size(),
    "registered owner": m[reg].groupby("boro").size(),
    "managing agent": m[m.agent.fillna("") != ""].groupby("boro").size(),
    "agent street address (company)": m[m.agent_co == 1].groupby("boro").size(),
    "owner street address (corporation)": m[m.owner_addr.fillna("") != ""].groupby("boro").size(),
})
cov.loc["New York City"] = cov.sum()
share = (cov.iloc[:, 1:].div(cov["ranked buildings"], axis=0) * 100).round(1)
share.insert(0, "ranked buildings", cov["ranked buildings"])
share

In [ ]:
unit = r"#|\b(?:apt|apartment|unit)\b"
checks = {
    "businessapartment column never pulled": "businessapartment" not in raw.columns,
    "no unit designator in an agent address": not c.agent_addr.str.contains(unit, case=False, regex=True).any(),
    "no unit designator in an owner address": not c.owner_addr.str.contains(unit, case=False, regex=True).any(),
    "agents filed as a person carry no street number": not c[(c.agent_co == 0) & (c.agent != "")]
        .agent_addr.str.contains(r"\d", regex=True).any(),
    "one row per bbl": c.bbl.is_unique,
    "bbl and registrationid are int64": str(c.bbl.dtype) == "int64" and str(c.registrationid.dtype) == "int64",
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values())

How concentrated is responsibility? Among the ranked buildings with a modeled winter risk
of 50% or more, the share held by the largest managing agents (counts only; this notebook
does not publish a list of names).

In [ ]:
hi = m[(m.p_season >= 0.5) & (m.agent.fillna("") != "")]
per = hi.groupby("agent").size().sort_values(ascending=False)
print(f"{len(m[m.p_season >= 0.5]):,} buildings at 50%+ risk; {len(hi):,} have a managing agent "
      f"on file, spread over {per.size:,} agents")
for k in (10, 50, 100):
    print(f"  the {k} agents with the most of them hold {per.head(k).sum():,} "
          f"({per.head(k).sum() / len(hi):.1%})")

## 3. Rebuild `public/data/` for the Vercel site with the new fields

In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, "web/build_assets.py"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0
print(f"build_assets: {time.time() - t0:.1f} s")

## 4. Every notice, drafted through the code the site serves

`app.building_payload()` and `app.draft_message()` are exactly what
`/api/draft_message/<bbl>` returns, in FULL mode as on this instance. Building 60119
(BBL 2043190010) is the running example in the submission.

In [ ]:
os.environ["APP_MODE"] = "full"
import app

p = app.building_payload(2043190010)
print({k: p[k] for k in ("season", "last_season", "landlord", "agent", "registration")})

In [ ]:
d = app.draft_message(p, "tenant", "2156 Cruger Avenue, Bronx, NY 10462")
print("To:", *d["recipients"], sep="\n  ")
print("Subject:", d["subject"], "\n")
print(d["body"])

In [ ]:
d = app.draft_message(p, "landlord", "2156 Cruger Avenue, Bronx, NY 10462")
print("To:", *d["recipients"], sep="\n  ")
print("Subject:", d["subject"], "\n")
print(d["body"])

In [ ]:
# both framings for all 63,727 ranked buildings, then the same checks on every draft
bbls = risk.bbl.tolist()
t0 = time.time()
payloads = [app.building_payload(b) for b in bbls]
t1 = time.time()
drafts = [app.draft_message(pl, role) for pl in payloads for role in app.DRAFT_ROLES]
t2 = time.time()
print(f"{len(payloads):,} payloads in {t1 - t0:.1f} s; {len(drafts):,} drafts in {t2 - t1:.1f} s "
      f"({len(drafts) / (t2 - t1):,.0f} drafts per second on {platform.machine()})")

import re
unit_rx = re.compile(r"#|\b(?:apt|unit)\b", re.I)
bad_unit = sum(bool(unit_rx.search(d["body"] + " ".join(d["recipients"]))) for d in drafts)
over_99 = sum("100%" in d["body"] for d in drafts)
system_signed = sum(d["body"].rstrip().endswith("HeatAhead") for d in drafts)
print("drafts with a unit designator:", bad_unit, "| quoting 100%:", over_99,
      "| signed by the system:", system_signed)
assert bad_unit == over_99 == system_signed == 0

## Result

- Section 2's table is the coverage the building page now shows: owner, managing agent and
  business address, with the HPD Online link for every registered building.
- `public/data/` was rebuilt here on s390x; copy it back and commit it with the code
  (`docs/VERCEL.md` section 2), then restart `heatahead-web` so the FULL app on this
  instance reads `bbl_contacts.csv` too.
- T5c (an AI-written draft) stays roadmap: it would replace `draft_message()` behind the
  same `/api/draft_message` response, keeping `note: "draft only"` and a human send step.